# 🔧 Feature Engineering

## Overview
This notebook transforms the raw data into a feature-rich dataset suitable for machine learning models. Key steps include:
1.  **Lag Features**: Capturing past sales performance.
2.  **Rolling Window Features**: Capturing trends and moving averages.
3.  **Time-Based Features**: Extracting seasonal patterns.
4.  **Price Features**: Analyzing price dynamics.
5.  **Interaction Features**: Combining key variables.
6.  **Data Saving**: Exporting the engineered dataset for model training.

In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)

print("=" * 60)
print("🔧 FEATURE ENGINEERING FOR DEMAND FORECASTING")
print("=" * 60)
print("✅ Libraries imported!")

In [ ]:
# Cell 2: Load Data
try:
    df = pd.read_csv('../data/raw/grocery_sales.csv')
    df['date'] = pd.to_datetime(df['date'])

    print(f"📊 Dataset Shape: {df.shape}")
    print(f"📅 Date Range: {df['date'].min()} to {df['date'].max()}")
    print(f"\nColumns: {list(df.columns)}")

    # Sort by date (CRITICAL for time series!)
    df = df.sort_values(['item_id', 'store_id', 'date']).reset_index(drop=True)
    print("\n✅ Data sorted by item, store, and date")
except FileNotFoundError:
    print("❌ Error: File not found. Please check '../data/raw/grocery_sales.csv'")

In [ ]:
# Cell 3: Lag Features (Most Important for Forecasting!)
print("🔄 Creating Lag Features...")
print("=" * 60)

# Create lag features for each item-store combination
lag_periods = [1, 7, 14, 28]  # Yesterday, last week, 2 weeks ago, 4 weeks ago

for lag in lag_periods:
    df[f'sales_lag_{lag}'] = df.groupby(['item_id', 'store_id'])['sales'].shift(lag)
    print(f"✅ Created lag_{lag} feature")

print("\nLag features created:")
print([col for col in df.columns if 'lag' in col])

In [ ]:
# Cell 4: Rolling Window Features (Moving Averages & Trends)
print("📊 Creating Rolling Window Features...")
print("=" * 60)

windows = [7, 14, 28]  # 1 week, 2 weeks, 4 weeks

for window in windows:
    # Rolling mean
    df[f'sales_rolling_mean_{window}'] = df.groupby(['item_id', 'store_id'])['sales'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    
    # Rolling std (volatility)
    df[f'sales_rolling_std_{window}'] = df.groupby(['item_id', 'store_id'])['sales'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).std()
    )
    
    # Rolling max
    df[f'sales_rolling_max_{window}'] = df.groupby(['item_id', 'store_id'])['sales'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).max()
    )
    
    # Rolling min
    df[f'sales_rolling_min_{window}'] = df.groupby(['item_id', 'store_id'])['sales'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).min()
    )
    
    print(f"✅ Created rolling features for window={window}")

print(f"\n✅ Total rolling features: {len([col for col in df.columns if 'rolling' in col])}")

In [ ]:
# Cell 5: Time-Based Features
print("📅 Creating Advanced Time Features...")
print("=" * 60)

# Week of year
df['week_of_year'] = df['date'].dt.isocalendar().week

# Quarter
df['quarter'] = df['date'].dt.quarter

# Is month start/end
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

# Days since start (trend)
df['days_since_start'] = (df['date'] - df['date'].min()).dt.days

# Day of month
df['day_of_month'] = df['date'].dt.day

# Cyclical encoding for month (sine/cosine)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Cyclical encoding for day of week
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

print("✅ Time-based features created:")
time_features = ['week_of_year', 'quarter', 'is_month_start', 'is_month_end', 
                 'days_since_start', 'day_of_month', 'month_sin', 'month_cos', 
                 'dow_sin', 'dow_cos']
for feat in time_features:
    print(f"   • {feat}")

In [ ]:
# Cell 6: Price Features
print("💰 Creating Price Features...")
print("=" * 60)

# Price change from base price
df['price_discount_pct'] = ((df['base_price'] - df['price']) / df['base_price']) * 100

# Price relative to category average
category_avg_price = df.groupby('category')['price'].transform('mean')
df['price_vs_category_avg'] = df['price'] / category_avg_price

print("✅ Price features created:")
print("   • price_discount_pct")
print("   • price_vs_category_avg")

In [ ]:
# Cell 7: Interaction Features
print("🔀 Creating Interaction Features...")
print("=" * 60)

# Weekend + Promotion interaction
df['weekend_promo'] = df['is_weekend'] * df['on_promotion']

# Holiday + Promotion interaction
df['holiday_promo'] = df['is_holiday'] * df['on_promotion']

# Category encoding (will be useful for models)
df['category_encoded'] = df['category'].astype('category').cat.codes

# Store type encoding
df['store_type_encoded'] = df['store_type'].astype('category').cat.codes

# Store size encoding
df['store_size_encoded'] = df['store_size'].astype('category').cat.codes

print("✅ Interaction features created")
print(f"\n📊 Total features now: {df.shape[1]}")

In [ ]:
# Cell 8: Handle Missing Values
print("🔍 Checking Missing Values...")
print("=" * 60)

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print("Missing values found (expected due to lag features):")
    print(missing)
    
    # Drop rows with missing lag features (first few days of each item-store)
    print(f"\nOriginal shape: {df.shape}")
    df = df.dropna()
    print(f"After dropping NaN: {df.shape}")
else:
    print("✅ No missing values!")

In [ ]:
# Cell 9: Visualizing Key Features
print("📊 Visualizing Key Features...")
print("=" * 60)

# Correlation of new features with sales
new_features = [col for col in df.columns if 'lag' in col or 'rolling' in col]
if new_features:
    corr_with_sales = df[new_features + ['sales']].corr()['sales'].sort_values(ascending=False)
    print("Top 10 Features Correlated with Sales:")
    print(corr_with_sales.head(10))
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x=corr_with_sales.head(10).index, y=corr_with_sales.head(10).values)
    plt.title('Top 10 Features Correlated with Sales')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Cell 10: Save Processed Data
print("💾 Saving Processed Data...")
print("=" * 60)

output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'features_engineered.csv')
df.to_csv(output_path, index=False)

print(f"✅ Data saved to {output_path}")
print(f"Shape: {df.shape}")